# Thực nghiệm PhoBERT: đánh giá bộ nhãn mở rộng

Notebook này chạy một cấu hình PhoBERT đã được chọn trên GPU Kaggle hoặc Colab. Nó dùng tệp nhãn đã hợp nhất, huấn luyện với toàn bộ dữ liệu nhưng chỉ đánh giá outer holdout trên các tầng lấy mẫu bảo toàn phân phối gốc.

## Trước khi chạy

1. Trên Kaggle, chọn **Settings → Accelerator → GPU** và bật **Internet**; trên Colab, chọn **Runtime → Change runtime type → T4 GPU**.
2. Tạo một phiên bản mới của Kaggle Dataset `phuocthoai/stock-trend-forecasting` có chứa `labeled_merged.csv` vừa được tạo cục bộ. Tệp này bị Git bỏ qua có chủ ý vì chứa dữ liệu nghiên cứu, không phải mã nguồn.
3. Notebook lấy mã từ branch `feat/stratified-sentiment-cv`; lấy dữ liệu từ Kaggle Dataset. Hai nguồn này tách biệt.
4. Không dùng nhãn còn ở trạng thái sơ bộ. Cell kiểm tra sẽ chặn tệp không đủ 1.306 nhãn đã được rà soát và thiếu thông tin xuất xứ.

In [ ]:
from pathlib import Path
import importlib
import importlib.util
import os
import subprocess
import sys

REPO_URL = "https://github.com/nphuoctho/stock-trend-forecasting.git"
BRANCH = "feat/stratified-sentiment-cv"
REPO_DIR = Path("/content/stock-trend-forecasting")
if Path("/kaggle").exists():
    REPO_DIR = Path("/kaggle/working/stock-trend-forecasting")

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
os.chdir(REPO_DIR)
SRC_DIR = REPO_DIR / "src"
if not SRC_DIR.is_dir():
    raise FileNotFoundError(f"Không tìm thấy source package: {SRC_DIR}")
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
SUBPROCESS_ENV = {
    **os.environ,
    "PYTHONPATH": os.pathsep.join(
        part for part in (str(SRC_DIR), os.environ.get("PYTHONPATH", "")) if part
    ),
}
# Do not replace the platform torch/CUDA build. This exact stack is recorded
# by the successful Kaggle CV manifest: torch 2.10.0+cu128 + transformers 5.15.1.
HF_PKGS = [
    "transformers==5.15.1",
    "tokenizers>=0.22,<=0.23.0",
    "huggingface-hub>=1.5,<2.0",
    "accelerate>=1.1,<2",
    "sentencepiece>=0.2,<1",
]
if importlib.util.find_spec("torchvision") is None:
    print("torchvision is not installed; text-only PhoBERT training continues.")
else:
    torchvision_check = subprocess.run(
        [sys.executable, "-c", "import torchvision"],
        capture_output=True,
        text=True,
    )
    if torchvision_check.returncode != 0:
        print("Removing incompatible torchvision; PhoBERT training is text-only.")
        subprocess.run(
            [sys.executable, "-m", "pip", "uninstall", "-y", "torchvision", "timm"],
            check=True,
        )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", *HF_PKGS],
    check=True,
)
importlib.invalidate_caches()
EXPECTED_TORCH = "2.10.0+cu128"
subprocess.run(
    [
        sys.executable,
        "-c",
        "import torch; expected='2.10.0+cu128'; actual=torch.__version__; assert actual == expected, f'PyTorch {actual}; cần {expected}. Khởi tạo phiên Kaggle mới; nếu vẫn khác, không chạy với stack chưa được kiểm chứng.'",
    ],
    check=True,
    env=SUBPROCESS_ENV,
)
subprocess.run(
    [
        sys.executable,
        "-c",
        "from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, Trainer, TrainingArguments; TrainingArguments(output_dir='/tmp/stf-check', eval_strategy='epoch', save_strategy='epoch', save_only_model=True, use_cpu=True)",
    ],
    check=True,
    env=SUBPROCESS_ENV,
)
import stf
MODEL_SOURCE = REPO_DIR / "src/stf/sentiment/model.py"
model_source = MODEL_SOURCE.read_text(encoding="utf-8")
if "save_only_model=True" not in model_source:
    raise RuntimeError(
        f"Branch {BRANCH} thiếu cấu hình checkpoint tiết kiệm dung lượng."
    )
print("Disk-safe CV source: enabled")
print("Repository ready:", REPO_DIR)
print("Package source:", stf.__file__)
print("Transformers runtime: 5.15.1")

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("GPU chưa được bật. Chọn T4 GPU rồi chạy lại cell này.")

In [ ]:
if Path("/kaggle").exists():
    DRIVE_ROOT = Path("/kaggle/working/stock-trend-experiments")
else:
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/stock-trend-experiments")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("Results will be saved under:", DRIVE_ROOT)

In [ ]:
KAGGLE_DATASET = "phuocthoai/stock-trend-forecasting"
ATTACHED_DIR = Path("/kaggle/input/stock-trend-forecasting")

if ATTACHED_DIR.exists():
    DATASET_DIR = ATTACHED_DIR
    print("Using attached Kaggle Dataset:", DATASET_DIR)
else:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=True
    )
    import kagglehub

    DATASET_DIR = Path(kagglehub.dataset_download(KAGGLE_DATASET))
    print("Downloaded Kaggle Dataset to:", DATASET_DIR)

matches = sorted(DATASET_DIR.rglob("labeled_merged.csv"))
if len(matches) != 1:
    raise FileNotFoundError(
        "Cần đúng một tệp labeled_merged.csv trong Kaggle Dataset đã cập nhật."
    )
DATA_PATH = matches[0]
print("Using reviewed merged label data:", DATA_PATH)

In [ ]:
import pandas as pd
from stf.sentiment.dataset import load_labeled

raw_labeled = pd.read_csv(DATA_PATH)
required = {
    "title", "body_preview", "label", "stratum", "usage",
    "annotation_source", "annotation_status",
}
missing = required - set(raw_labeled.columns)
assert not missing, f"Thiếu cột bắt buộc: {sorted(missing)}"
assert len(raw_labeled) == 1306, f"Cần 1.306 nhãn đã hợp nhất, nhận {len(raw_labeled)}."
assert raw_labeled["annotation_source"].eq("human_reviewed").all(), (
    "Tệp còn nhãn chưa được người rà soát xác nhận."
)
assert raw_labeled["annotation_status"].eq("REVIEWED").all(), (
    "Tệp còn trạng thái nhãn sơ bộ."
)
assert set(raw_labeled["usage"]).issubset({"eval_or_train", "train_only"})

labeled = load_labeled(DATA_PATH)
print("Rows:", len(labeled))
print("Label distribution:")
print(labeled["label"].value_counts().to_string())
print("Evaluable rows:", int((raw_labeled["usage"] == "eval_or_train").sum()))
print("Train-only enriched rows:", int((raw_labeled["usage"] == "train_only").sum()))

## Cấu hình chạy

Cấu hình này đánh giá lại nhánh `title_context` + `head_tail` với 5 epoch và trọng số tần suất nghịch. Mỗi outer holdout chỉ lấy từ `eval_random` và `baseline_random`; các mẫu làm giàu lớp thiểu số chỉ tham gia huấn luyện.

In [ ]:
INPUT_VARIANT = "title_context"
TRUNCATION_STRATEGY = "head_tail"
CLASS_WEIGHTING = "inverse_frequency"
EVAL_STRATA = ("eval_random", "baseline_random")
FOLDS = 5
EPOCHS = 5
BATCH_SIZE = 16
SEED = 42
RUN_NAME = f"merged__{INPUT_VARIANT}__{TRUNCATION_STRATEGY}__{CLASS_WEIGHTING}__e{EPOCHS}"
OUTPUT_DIR = DRIVE_ROOT / RUN_NAME

command = [
    sys.executable,
    "-m",
    "stf.cli",
    "sentiment-cv",
    "--data",
    str(DATA_PATH),
    "--input-variant",
    INPUT_VARIANT,
    "--truncation-strategy",
    TRUNCATION_STRATEGY,
    "--class-weighting",
    CLASS_WEIGHTING,
    "--folds",
    str(FOLDS),
    "--epochs",
    str(EPOCHS),
    "--batch-size",
    str(BATCH_SIZE),
    "--seed",
    str(SEED),
    "--eval-strata",
    *EVAL_STRATA,
    "--output",
    str(OUTPUT_DIR),
]
print(" ".join(command))
subprocess.run(command, check=True, cwd=REPO_DIR, env=SUBPROCESS_ENV)

In [ ]:
import json

cv_json = OUTPUT_DIR / "cv_results.json"
cv_csv = OUTPUT_DIR / "cv_results.csv"
result = json.loads(cv_json.read_text(encoding="utf-8"))
folds = pd.read_csv(cv_csv)
print("Fold results:")
display(folds)
print("Aggregate:")
display(pd.DataFrame(result["aggregate"]).T)
print("Saved to:", OUTPUT_DIR)

## Đọc kết quả

- Đọc `macro_f1`, `balanced_accuracy`, F1 từng lớp từ `cv_results.json` và `cv_results.csv`; không kết luận từ accuracy đơn lẻ.
- Năm outer holdout cộng lại có 656 dòng, trung bình khoảng 131 dòng mỗi fold, gồm tổng cộng 59 nhãn `NEGATIVE`; khoảng tin cậy của recall lớp này rộng hơn bộ đánh giá toàn bộ vì nó bảo toàn phân phối gốc.
- `sentiment-cv` không sinh ma trận nhầm lẫn hay dự đoán từng dòng. Không yêu cầu các artifact này khi diễn giải lần chạy.
- So sánh với kết quả 306 nhãn đã có. Chỉ khi kết quả mới cải thiện bền vững trên các fold mới cân nhắc chạy lại nhánh dự báo giá.
- Lưu nguyên thư mục `OUTPUT_DIR`; `cv_results.json`, `cv_results.csv` và manifest từng fold là bằng chứng tái lập.

## Nén artifact để tải về

Chạy ô này sau khi có kết quả. Tệp `.zip` được tạo cạnh `OUTPUT_DIR`, nên không tự nằm trong thư mục nguồn đang nén.

In [ ]:
import shutil

archive_path = Path(
    shutil.make_archive(
        base_name=str(OUTPUT_DIR),
        format="zip",
        root_dir=OUTPUT_DIR.parent,
        base_dir=OUTPUT_DIR.name,
    )
)
print(f"Archive ready: {archive_path}")